# Multiscale Gaussian Derivative Orientation

This runs first-derivative Gaussian filters at several `sigma` values and steers each `gradient_x`, `gradient_y` pair.

## How It Works

A small `sigma` reacts to narrow details and noise. A large `sigma` gives a wider, smoother edge response. Each scale is still first order because it uses only horizontal and vertical first derivatives.

### Reference

Canny's 1986 edge detector is a common reference for first-derivative Gaussian edge detection. DOI [10.1109/TPAMI.1986.4767851](https://doi.org/10.1109/TPAMI.1986.4767851).

## Setup

These notebooks are split by how the orientation response is made. This one first computes `gradient_x` and `gradient_y`, then projects that gradient pair onto an angle.


In [ ]:
# Imports.
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import torch

from agfb_filters import multiscale_gaussian_derivative_orientation_banks

## Filter Settings

Change `sigmas` to compare different derivative scales. `angle_index` chooses which orientation to plot for every scale.

In [ ]:
# Choose scales and angles.
sigmas = (1.0, 3.0, 7.0)
angle_count = 12
angle_index = 0

## Test Image

The default image is a 1024 by 1024 black-to-white step edge. The commented lines show the smallest path-based image import you can use instead.


In [ ]:
# Build the test image.
size = 1024
image = torch.zeros(1, size, size)
image[:, :, size // 2 :] = 1.0

# To use your own image instead, uncomment these lines.
# image_path = "path/to/image.png"
# image = torch.as_tensor(plt.imread(image_path), dtype=torch.float32)
# if image.ndim == 3:
#     image = image[..., :3].mean(dim=-1)
# image = image.unsqueeze(0)

plt.figure(figsize=(4, 4))
plt.imshow(image[0], cmap="gray", vmin=0, vmax=1)
plt.title("input image")
plt.axis("off")

## Run And Plot

Each result is one orientation bank for one `sigma`. The plot shows the same angle at every scale.

In [ ]:
# Time the multiscale call.
start = time.perf_counter()
results = multiscale_gaussian_derivative_orientation_banks(
    image, sigmas=sigmas, angle_count=angle_count
)
elapsed = time.perf_counter() - start
print(f"{elapsed:.4f} seconds")

limit = max(float(result.responses.abs().max()) for result in results)
fig, axes = plt.subplots(1, len(results), figsize=(3 * len(results), 3))
for axis, sigma, result in zip(axes, sigmas, results, strict=True):
    axis.imshow(result.responses[0, angle_index], cmap="gray", vmin=-limit, vmax=limit)
    axis.set_title(f"sigma {sigma}")
    axis.axis("off")
plt.tight_layout()